In [ ]:
# Lecture 21 — Partitioning: repartition vs coalesce
# Spark 101 · Module 7 · Lecture 21
#
# Import this file into a Databricks workspace as a notebook.
# Databricks recognizes `# COMMAND ----------` as a cell boundary on import.
# In VS Code / PyCharm this stays a plain Python file — same source, two homes.

In [ ]:
print("Spark version:", spark.version)
# Databricks gives you `spark` for free — no SparkSession.builder needed.

In [ ]:
# --- How many partitions do I have? Always check first. --------------------
df = spark.range(0, 1_000_000)
print("start partitions:", df.rdd.getNumPartitions())
# On a cluster this reflects default parallelism (roughly your core count).

In [ ]:
# --- repartition: full shuffle into N EVEN partitions ----------------------
more = df.repartition(16)
print("after repartition(16):", more.rdd.getNumPartitions())

# Check the balance — repartition spreads rows evenly.
from pyspark.sql.functions import spark_partition_id, count
(
    more.withColumn("pid", spark_partition_id())
        .groupBy("pid").agg(count("*").alias("rows"))
        .orderBy("pid")
        .show(20)
)

In [ ]:
# --- coalesce: merge DOWN with no full shuffle -----------------------------
fewer = more.coalesce(4)
print("after coalesce(4):", fewer.rdd.getNumPartitions())

# coalesce only reduces — asking for MORE than you have does nothing:
print("coalesce(64) on 4 parts:", fewer.coalesce(64).rdd.getNumPartitions())  # still 4
# To actually go up, you need repartition (a shuffle):
print("repartition(64):", fewer.repartition(64).rdd.getNumPartitions())        # 64

In [ ]:
# --- See it in the plan: repartition = Exchange, coalesce = no Exchange -----
print("=== repartition(16) plan (has Exchange) ===")
df.repartition(16).explain()

print("=== coalesce(4) plan (no full-shuffle Exchange) ===")
more.coalesce(4).explain()

In [ ]:
# --- Practical: coalesce before writing to control file count (L19) --------
import os
base = "/tmp/spark101_l21"

# 16 partitions -> 16 output files
more.write.mode("overwrite").parquet(f"file:{base}/many")
print("many:", len([f for f in os.listdir(f"{base}/many") if f.endswith('.parquet')]))

# coalesce(2) -> 2 output files, no full shuffle
more.coalesce(2).write.mode("overwrite").parquet(f"file:{base}/few")
print("few:", len([f for f in os.listdir(f"{base}/few") if f.endswith('.parquet')]))

In [ ]:
# Repartition by a COLUMN — hash-partition rows by key value.
from pyspark.sql.functions import col
keyed = spark.range(0, 1_000_000).withColumn("key", (col("id") % 5))
print("partitions:", keyed.repartition(5, "key").rdd.getNumPartitions())

In [ ]:
# Skew: a hot key makes one partition far bigger than the rest.
from pyspark.sql.functions import when, spark_partition_id, count
skewed = spark.range(0, 1_000_000).withColumn(
    "key", when(col("id") % 10 == 0, 0).otherwise(col("id") % 100))
(skewed.repartition(8, "key")
       .withColumn("pid", spark_partition_id())
       .groupBy("pid").agg(count("*").alias("rows")).orderBy("pid").show())

In [ ]:
# The shuffle-partition count every wide op uses by default (200).
print("shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
spark.conf.set("spark.sql.shuffle.partitions", 8)   # tune down for small data
print("now:", spark.conf.get("spark.sql.shuffle.partitions"))

In [ ]:
# Recap:
#   - Check first: df.rdd.getNumPartitions().
#   - repartition(N): full shuffle, even sizes, up OR down — before heavy work.
#   - coalesce(N): cheap merge, down ONLY, maybe uneven — before writing.
#   - Wrong partition count = idle cores or overhead = the #1 slow-job cause.
# Next lecture: the shuffle itself.